# Step 4b · Train the state reader on Kaggle (v1: configs/reader.yaml, v2/v3/v4: reader_v2/v3/v4.yaml)

Settings: **Accelerator → GPU T4 x2 (or P100)**, **Internet → On**, **Add Input → Datasets → Your Datasets → robot-world-model-dataset** (and, for v4, the DAgger dataset).

W&B: **Add-ons → Secrets → Add secret** named `WANDB_API_KEY` (value from wandb.ai/authorize), and tick it for this notebook.

Download `reader.zip` from the output panel afterwards: ranking B runs on the laptop.

In [ ]:
# 1. Fresh clone of the repo
%cd /kaggle/working
!rm -rf robot-world-model
!git clone -q https://github.com/Swayam-27/robot-world-model.git
%cd /kaggle/working/robot-world-model
!git log --oneline -1

In [ ]:
# 2. W&B login from the Kaggle secret (no MuJoCo needed: the reader trains on stored frames)
!pip install -q wandb
import os
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("W&B key loaded")

In [ ]:
# 3. Find the uploaded datasets (folders that hold episodes.csv). The DAgger one (reader v4) has 'dagger' in its path
import glob
hits = [os.path.dirname(h) for h in glob.glob("/kaggle/input/**/episodes.csv", recursive=True)]
main = [h for h in hits if "dagger" not in h.lower()]
assert len(main) == 1, f"expected one main dataset, found {main}: Add Input -> Datasets -> Your Datasets"
DATA = main[0]; DAGGER = next((h for h in hits if "dagger" in h.lower()), "")
print("main:", DATA, len(glob.glob(DATA + "/ep_*.npz")), "episodes | dagger:", DAGGER or "not attached")

In [ ]:
# 4. Train (about 1 minute to load the data, then a few minutes per epoch). Pick the version here.
CONFIG = "configs/reader_v4.yaml"   # v4 = v1 + DAgger episodes (needs the dagger dataset attached); v1-v3: reader.yaml, reader_v2.yaml, reader_v3.yaml
EXTRA = f'--extra-data-dir "{DAGGER}"' if "v4" in CONFIG else ""
assert "v4" not in CONFIG or DAGGER, "reader v4 needs the dagger dataset: Add Input -> Datasets -> Your Datasets"
!python -m models.train_reader --config $CONFIG --data-dir "$DATA" $EXTRA

In [ ]:
# 5. Pack the results for the laptop (download reader.zip from the Output panel, unzip into outputs/)
!cd outputs && zip -qr /kaggle/working/reader.zip reader* && unzip -l /kaggle/working/reader.zip